# 07 -- Async Operations

## What We'll Cover
1. AsyncSupermemory client
2. Concurrent ingestion (10x+ speedup)
3. Parallel profile lookups
4. Async file uploads
5. Error handling in async mode
6. Performance comparison: sync vs async


In [ ]:
# 7.1 Sync vs Async Clients
import asyncio
from supermemory import Supermemory, AsyncSupermemory

# Sync client (what we've used so far)
sync_client = Supermemory()

# Async client -- identical API, but all methods are awaitable
async_client = AsyncSupermemory()

print("Sync client:", type(sync_client).__name__)
print("Async client:", type(async_client).__name__)
print("Functionality is identical -- only the calling pattern differs.")


In [ ]:
# 7.2 Basic async usage pattern
async def basic_async_example():
    """Demonstrate basic async pattern."""
    async with AsyncSupermemory() as client:
        # Add a memory
        response = await client.add(
            content="Async operations are more efficient for bulk ingestion.",
            container_tag="async_demo"
        )
        print(f"Added: {response.id}")

        # Search
        results = await client.search.execute(q="async operations")
        print(f"Search results: {len(results.results)}")

# Run it
asyncio.run(basic_async_example())


In [ ]:
# 7.3 Concurrent ingestion -- the killer async feature
# Process many documents in parallel, not sequentially
import time

async def concurrent_ingestion():
    """Add 20 documents concurrently."""
    async with AsyncSupermemory() as client:
        # Prepare 20 documents
        docs = [
            f"Document {i}: This is content about topic {i} with some details."
            for i in range(20)
        ]

        # Launch all add operations concurrently
        tasks = [
            client.add(content=doc, container_tag="bulk_test")
            for doc in docs
        ]

        # Wait for all to complete
        results = await asyncio.gather(*tasks)

        print(f"Ingested {len(results)} documents concurrently!")
        for r in results[:5]:
            print(f"  {r.id}: {r.status}")
        print(f"  ... and {len(results) - 5} more")
        return results

# Compare timing
start = time.time()
asyncio.run(concurrent_ingestion())
print(f"\nTime: {time.time() - start:.2f}s (20 concurrent adds)")
print("Sequential would take ~20x longer!")


In [ ]:
# 7.4 Parallel profile lookups -- perfect for multi-tenant apps
async def parallel_profiles(user_ids):
    """Look up profiles for multiple users in parallel."""
    async with AsyncSupermemory() as client:
        tasks = [
            client.profile(container_tag=f"user_{uid}")
            for uid in user_ids
        ]
        profiles = await asyncio.gather(*tasks)

        for uid, profile in zip(user_ids, profiles):
            static_count = len(profile.profile.static)
            dynamic_count = len(profile.profile.dynamic)
            print(f"user_{uid}: {static_count} static, {dynamic_count} dynamic facts")

        return dict(zip(user_ids, profiles))

asyncio.run(parallel_profiles(["alex", "sarah", "alice", "bob"]))


In [ ]:
# 7.5 Async error handling with retries
from supermemory import RateLimitError, APIConnectionError

async def robust_add(client, content, container_tag, max_retries=3):
    """Add a memory with retry logic for transient errors."""
    for attempt in range(max_retries):
        try:
            return await client.add(content=content, container_tag=container_tag)
        except RateLimitError:
            wait = 2 ** (attempt + 1)  # Exponential backoff: 2, 4, 8 seconds
            print(f"Rate limited, retrying in {wait}s (attempt {attempt+1}/{max_retries})...")
            await asyncio.sleep(wait)
        except APIConnectionError as e:
            print(f"Connection error: {e}, retrying...")
            await asyncio.sleep(1)
    raise Exception(f"Failed after {max_retries} attempts")

async def demo_robust():
    async with AsyncSupermemory() as client:
        r = await robust_add(client, "Important data that must be stored.", "critical")
        print(f"Stored: {r.id}")

asyncio.run(demo_robust())


## 7.6 Performance Comparison

| Operation | Sync | Async (concurrent) |
|-----------|------|--------------------|
| 1 document | ~100ms | ~100ms |
| 10 documents | ~1000ms | ~150ms |
| 100 documents | ~10s | ~500ms |
| 10 user profiles | ~500ms | ~80ms |

**Rule of thumb:** Use async when dealing with 3+ independent operations.

## 7.7 Key Takeaways

- `AsyncSupermemory` has the same API as `Supermemory` -- just add `await`
- Use `asyncio.gather()` for concurrent operations (10x+ speedup)
- `async with` ensures proper cleanup of HTTP connections
- Error handling works the same -- catch `RateLimitError`, `APIConnectionError`, etc.

**Next:** Notebook 08 -- Self-Hosting Supermemory
